# SOCKETS — TCP client/server (9569 P2)

This is the last P2 boilerplate. Everything else is already in POOP, S&S, ADT, DB, WEB.

**Do not run a blocking `accept()` inside this notebook.** Sockets are two programs. Files on disk:

- `sockets_server.py`
- `sockets_client.py`

Two terminals, **this folder**, server first:

```text
python sockets_server.py
python sockets_client.py
```

Demo secret word: `apple`. Type `QUIT` to stop. Port `5000` — do not run Flask at the same time.

Your old `sockets client.py` / `sockets server.py` had exam-killing bugs (`=` vs `==`, undefined `secret_word` / `client_socket`). The underscored filenames are the cleaned templates.


## Line receiver — memorise this

`recv(1024)` is not “one message”. Loop until `\n`.


In [ ]:
def recv_line(sock):
    data = b""
    while b"\n" not in data:
        chunk = sock.recv(1024)
        if not chunk:
            break
        data += chunk
    return data.decode().strip()


## Server (also `sockets_server.py`)


In [ ]:
# server: socket -> bind -> listen -> accept -> loop -> close
import socket

def recv_line(sock):
    data = b""
    while b"\n" not in data:
        chunk = sock.recv(1024)
        if not chunk:
            break
        data += chunk
    return data.decode().strip()

SECRET = "apple"
server = socket.socket()
server.bind(("127.0.0.1", 5000))
server.listen()
print("waiting for client...")
client, address = server.accept()
print("connected to", address)

while True:
    guess = recv_line(client)
    if guess == "QUIT":
        break
    elif guess == SECRET:
        client.sendall(b"WIN\n")
        break
    else:
        client.sendall(b"WRONG\n")

client.close()
server.close()


## Client (also `sockets_client.py`)


In [ ]:
# client: socket -> connect -> send/recv loop -> close
import socket

def recv_line(sock):
    data = b""
    while b"\n" not in data:
        chunk = sock.recv(1024)
        if not chunk:
            break
        data += chunk
    return data.decode().strip()

client = socket.socket()
client.connect(("127.0.0.1", 5000))
state = "WRONG"

while state == "WRONG":
    guess = input("key in your guess: ")
    client.sendall((guess + "\n").encode())
    if guess == "QUIT":
        break
    state = recv_line(client)
    if state == "WIN":
        print("you won")
        break
    print("try again")

print("program has ended")
client.close()


## Cheat-sheet

```text
server: bind((ip,port)); listen(); client,addr = accept()
client: connect((ip,port))
sendall(msg.encode())   # always end with \n
recv until b"\n" in buffer
close client, then server
```

| Bug | Fix |
|---|---|
| `while state = "WRONG"` | `==` |
| `client_socket.sendall` but socket named `client` | one name |
| Flask + sockets on 5000 | stop one of them |

Run order: server waiting → client connect → messages with `\n` → close both.
